In [ ]:
import pulp


class mipconv:
    _counter = 0

    @classmethod
    def _new_name(cls, prefix: str) -> str:
        cls._counter += 1
        return f"{prefix}_{cls._counter}"

    @staticmethod
    def _get_bounds(var):
        lb = var.lowBound
        ub = var.upBound
        if lb is None or ub is None:
            raise ValueError(
                f"Variable {var.name} must have finite lowBound and upBound "
                f"to linearize max/min safely."
            )
        return lb, ub

    @classmethod
    def max(cls, x, y, prob: pulp.LpProblem, name: str = None):
        """
        Create z = max(x, y) and add the corresponding MILP constraints to prob.

        Returns
        -------
        z : pulp.LpVariable
            Variable representing max(x, y)
        """
        if name is None:
            name = cls._new_name("max")

        x_lb, x_ub = cls._get_bounds(x)
        y_lb, y_ub = cls._get_bounds(y)

        # z bounds
        z_lb = max(x_lb, y_lb)
        z_ub = max(x_ub, y_ub)

        z = pulp.LpVariable(name, lowBound=z_lb, upBound=z_ub, cat="Continuous")
        b = pulp.LpVariable(f"{name}_bin", cat="Binary")

        # Tight M values from variable bounds
        # Need bounds for x - y and y - x
        M_xy = x_ub - y_lb   # upper bound on x - y
        M_yx = y_ub - x_lb   # upper bound on y - x

        # z >= x, z >= y
        prob += z >= x, f"{name}_ge_x"
        prob += z >= y, f"{name}_ge_y"

        # If b = 1 => z <= x
        # If b = 0 => relaxed by M_xy
        prob += z <= x + M_yx * (1 - b), f"{name}_le_x_branch"

        # If b = 0 => z <= y
        # If b = 1 => relaxed by M_xy
        prob += z <= y + M_xy * b, f"{name}_le_y_branch"

        return z

    # test the class with a simple example
if __name__ == "__main__":
    prob = pulp.LpProblem("MaxExample", pulp.LpMaximize)
    x = pulp.LpVariable("x", lowBound=0, upBound=10, cat="Continuous")
    y = pulp.LpVariable("y", lowBound=0, upBound=10, cat="Continuous")
    z = mipconv.max(x, y, prob)
    prob += z, "Objective"
    prob.solve()
    print(f"x: {x.varValue}, y: {y.varValue}, z: {z.varValue}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /data/dev/simplinho/.venv/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/d8852bfcf1bc417b9ff2795f7f3d7ace-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/d8852bfcf1bc417b9ff2795f7f3d7ace-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 23 RHS
At line 28 BOUNDS
At line 33 ENDATA
Problem MODEL has 4 rows, 4 columns and 10 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 10 - 0.00 seconds
Cgl0004I processed model has 4 rows, 4 columns (1 integer (1 of which binary)) and 10 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -10
Cbc0038I Relaxing continuous gives -10
Cbc0038I Before mini branch and bound, 1 integers at bound fixed and 3 continuous
Cbc0038I Mini branch and bound did no